In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json, random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from thop import profile as thop_profile

# ============================================================
# Evaluation-only script — loads the ALREADY TRAINED weights
# produced by the HAM10000 baseline training script and
# recomputes accuracy / precision / recall / f1, FLOPs, params,
# disk size, and CPU/GPU inference timing. No training happens
# here. Mirrors the CIFAR-10 "load saved model" eval script.
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Must match the training script exactly ─────────────────────
SEED        = 42
NUM_CLASSES = 7
IMG_SIZE    = 224
VAL_SPLIT   = 0.2
BATCH_SIZE  = 64

DATA_ROOT = "./ham10000"
CSV_PATH  = os.path.join(DATA_ROOT, "HAM10000_metadata.csv")
IMG_DIR   = os.path.join(DATA_ROOT, "images")

# Path to the weights saved by the training script
SAVE_PATH = "__ham10000__baseline_resnet50.pth"

HAM10000_CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Using device: {DEVICE}")

# ── Rebuild model architecture (must match training) ───────────
def build_model(num_classes=NUM_CLASSES):
    model    = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_model(NUM_CLASSES)
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model = model.to(DEVICE).eval()
print(f"✓ Weights loaded from {SAVE_PATH}")

# ── Dataset (same class used at training time) ──────────────────
class HAM10000Dataset(Dataset):
    def __init__(self, df, img_dir, transform=None, class_list=HAM10000_CLASSES):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.class2idx = {cls: i for i, cls in enumerate(class_list)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"] + ".jpg")
        image    = Image.open(img_path).convert("RGB")
        label    = self.class2idx[row["dx"]]
        if self.transform:
            image = self.transform(image)
        return image, label

# ── Rebuild the SAME validation split used at training time ─────
# Same CSV, same SEED, same stratify column, same test_size as the
# training script — this reconstructs the identical held-out set
# the saved weights were validated on.
df = pd.read_csv(CSV_PATH)
df = df[["image_id", "dx"]].drop_duplicates(subset="image_id").reset_index(drop=True)

_, val_df = train_test_split(
    df, test_size=VAL_SPLIT, random_state=SEED, stratify=df["dx"]
)
print(f"Validation set: {len(val_df)} images")

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_set    = HAM10000Dataset(val_df, IMG_DIR, transform=transform_val)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Classification metrics ───────────────────────────────────────
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(DEVICE)
        all_preds.extend(model(inputs).argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0)

# ── FLOPs ─────────────────────────────────────────────────────
dummy_flops = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
macs, _     = thop_profile(model, inputs=(dummy_flops,), verbose=False)
flops_G     = macs * 2 / 1e9

# ── Parameters & disk size ───────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
size_mb      = os.path.getsize(SAVE_PATH) / 1e6

# # ── Inference time — CPU (run FIRST, before GPU timing) ─────────
# print("\n⏱  Measuring CPU inference times ...")
# model_cpu = build_model(NUM_CLASSES)
# model_cpu.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
# model_cpu = model_cpu.eval()

# dummy_single_cpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
# dummy_batch_cpu  = torch.randn(128, 3, IMG_SIZE, IMG_SIZE)

# with torch.no_grad():
#     for _ in range(10):
#         model_cpu(dummy_single_cpu)

# times_cpu_single = []
# with torch.no_grad():
#     for _ in range(100):
#         t0 = time.perf_counter()
#         model_cpu(dummy_single_cpu)
#         times_cpu_single.append(time.perf_counter() - t0)
# inf_ms_single_cpu = np.mean(times_cpu_single) * 1000

# with torch.no_grad():
#     for _ in range(5):
#         model_cpu(dummy_batch_cpu)

# times_cpu_batch = []
# with torch.no_grad():
#     for _ in range(20):
#         t0 = time.perf_counter()
#         model_cpu(dummy_batch_cpu)
#         times_cpu_batch.append(time.perf_counter() - t0)
# inf_ms_batch128_cpu     = np.mean(times_cpu_batch) * 1000
# inf_ms_per_img_cpu      = inf_ms_batch128_cpu / 128
# throughput_imgs_sec_cpu = 128 / (inf_ms_batch128_cpu / 1000)

# del model_cpu, dummy_single_cpu, dummy_batch_cpu
# print("✓ CPU timing done")

# ── Inference time — GPU (falls back to CPU-style timing if no CUDA) ──
for i in range(6):
    use_cuda = DEVICE.type == "cuda"
    print("⏱  Measuring GPU inference times ..." if use_cuda
        else "⏱  No CUDA device found — measuring 'GPU' timings on CPU instead ...")

    dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    with torch.no_grad():
        for _ in range(50):
            model(dummy_single_gpu)
    if use_cuda:
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if use_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single_gpu)
            if use_cuda:
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    inf_ms_single_gpu = np.mean(times) * 1000

    dummy_batch_gpu = torch.randn(128, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    if use_cuda:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)

        with torch.no_grad():
            for _ in range(10):
                model(dummy_batch_gpu)
        torch.cuda.synchronize()

        batch_times_gpu = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch_gpu)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times_gpu.append(start_ev.elapsed_time(end_ev))
        inf_ms_batch128_gpu = np.mean(batch_times_gpu)
        gpu_timing_method = "CUDA events + torch.cuda.synchronize()"
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy_batch_gpu)
        batch_times_gpu = []
        with torch.no_grad():
            for _ in range(20):
                t0 = time.perf_counter()
                model(dummy_batch_gpu)
                batch_times_gpu.append((time.perf_counter() - t0) * 1000)
        inf_ms_batch128_gpu = np.mean(batch_times_gpu)
        gpu_timing_method = "time.perf_counter() (no CUDA available)"

    inf_ms_per_img_gpu      = inf_ms_batch128_gpu / 128
    throughput_imgs_sec_gpu = 128 / (inf_ms_batch128_gpu / 1000)
    print(f"✓ GPU timing done for {i}")
    print(f"  --- Inference (GPU) ---")
    print(f"  Single image GPU  : {inf_ms_single_gpu:.3f} ms")
    print(f"  Batch-128 GPU     : {inf_ms_batch128_gpu:.2f} ms")
    print(f"  Per-image GPU     : {inf_ms_per_img_gpu:.3f} ms")
    print(f"  Throughput GPU    : {throughput_imgs_sec_gpu:.1f} imgs/sec")
    print("\n")
print("✓ Loop for GPU timing is done")

# ── Print results ─────────────────────────────────────────────
print(f"\n{'='*55}")
print("BASELINE METRICS (recomputed from saved weights)")
print(f"{'='*55}")
print(f"  Accuracy          : {acc:.4f}")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")
print(f"  Parameters        : {total_params:,}")
print(f"  Model size        : {size_mb:.2f} MB")
print(f"  FLOPs             : {flops_G:.3f} GFLOPs")
# print(f"  --- Inference (CPU) ---")
# print(f"  Single image CPU  : {inf_ms_single_cpu:.3f} ms")
# print(f"  Batch-128 CPU     : {inf_ms_batch128_cpu:.2f} ms")
# print(f"  Per-image CPU     : {inf_ms_per_img_cpu:.3f} ms")
# print(f"  Throughput CPU    : {throughput_imgs_sec_cpu:.1f} imgs/sec")
print(f"  --- Inference (GPU) ---")
print(f"  Single image GPU  : {inf_ms_single_gpu:.3f} ms")
print(f"  Batch-128 GPU     : {inf_ms_batch128_gpu:.2f} ms")
print(f"  Per-image GPU     : {inf_ms_per_img_gpu:.3f} ms")
print(f"  Throughput GPU    : {throughput_imgs_sec_gpu:.1f} imgs/sec")

# ── Save updated JSON ─────────────────────────────────────────
baseline_metrics = {
    "accuracy"  : acc,
    "precision" : precision,
    "recall"    : recall,
    "f1"        : f1,
    "params"    : total_params,
    "size_mb"   : size_mb,
    "flops_G"   : flops_G,
    "inference_ms": {
        # "cpu": {
        #     "single_img_ms"      : round(inf_ms_single_cpu, 4),
        #     "batch128_ms"        : round(inf_ms_batch128_cpu, 4),
        #     "per_img_ms"         : round(inf_ms_per_img_cpu, 4),
        #     "throughput_imgs_sec": round(throughput_imgs_sec_cpu, 1),
        #     "timing_method"      : "time.perf_counter()",
        # },
        "gpu": {
            "single_img_ms"      : round(inf_ms_single_gpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_gpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_gpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_gpu, 1),
            "timing_method"      : gpu_timing_method,
        },
    },
}
with open("__ham10000__baseline_metrics_v2.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)
print(f"\n✓ Updated metrics saved → __ham10000__baseline_metrics_v2.json")

Using device: cuda
✓ Weights loaded from __ham10000__baseline_resnet50.pth
Validation set: 2003 images
⏱  Measuring GPU inference times ...
✓ GPU timing done for 0
  --- Inference (GPU) ---
  Single image GPU  : 4.563 ms
  Batch-128 GPU     : 161.26 ms
  Per-image GPU     : 1.260 ms
  Throughput GPU    : 793.8 imgs/sec


⏱  Measuring GPU inference times ...
✓ GPU timing done for 1
  --- Inference (GPU) ---
  Single image GPU  : 4.527 ms
  Batch-128 GPU     : 161.52 ms
  Per-image GPU     : 1.262 ms
  Throughput GPU    : 792.5 imgs/sec


⏱  Measuring GPU inference times ...
✓ GPU timing done for 2
  --- Inference (GPU) ---
  Single image GPU  : 4.584 ms
  Batch-128 GPU     : 161.63 ms
  Per-image GPU     : 1.263 ms
  Throughput GPU    : 791.9 imgs/sec


⏱  Measuring GPU inference times ...
✓ GPU timing done for 3
  --- Inference (GPU) ---
  Single image GPU  : 5.126 ms
  Batch-128 GPU     : 161.46 ms
  Per-image GPU     : 1.261 ms
  Throughput GPU    : 792.8 imgs/sec


⏱  Measuring GPU 